# 1. Complete Implementation with Heapq

In [1]:
import heapq
import time
import random
from dataclasses import dataclass
from typing import List, Tuple, Optional
from enum import Enum

class OrderType(Enum):
    BID = "BID"
    ASK = "ASK"

@dataclass
class Order:
    order_id: int
    price: float
    quantity: int
    order_type: OrderType

    def __lt__(self, other):
        """For heap ordering based on price priority then time priority"""
        if self.price == other.price:
            return self.order_id < other.order_id
        return self.price < other.price

class HeapOrderBook:
    def __init__(self):
        # For bids: use negative price to create max-heap
        self.bids = []  # Max-heap (store -price)
        self.asks = []  # Min-heap
        self.order_map = {}  # order_id -> (order, heap_reference)
        self.next_order_id = 1

    def add_order(self, price: float, quantity: int, order_type: OrderType) -> int:
        """Add an order and return order_id"""
        order_id = self.next_order_id
        self.next_order_id += 1

        order = Order(order_id, price, quantity, order_type)

        if order_type == OrderType.BID:
            # For max-heap, store (-price, order_id, order)
            heapq.heappush(self.bids, (-price, order_id, order))
        else:
            # For min-heap, store (price, order_id, order)
            heapq.heappush(self.asks, (price, order_id, order))

        self.order_map[order_id] = (order, order_type)
        self._match_orders()
        return order_id

    def _match_orders(self):
        """Match top bid and ask if possible"""
        while self.bids and self.asks:
            # Get top bid (highest price) and top ask (lowest price)
            neg_bid_price, bid_id, bid_order = self.bids[0]
            ask_price, ask_id, ask_order = self.asks[0]
            bid_price = -neg_bid_price  # Convert back to positive

            if bid_price >= ask_price:
                # Orders can match
                matched_quantity = min(bid_order.quantity, ask_order.quantity)

                # Update quantities
                bid_order.quantity -= matched_quantity
                ask_order.quantity -= matched_quantity

                # Remove if fully filled
                if bid_order.quantity == 0:
                    heapq.heappop(self.bids)
                    del self.order_map[bid_id]

                if ask_order.quantity == 0:
                    heapq.heappop(self.asks)
                    del self.order_map[ask_id]

            else:
                break

    def get_best_bid(self) -> Optional[float]:
        """Get highest bid price"""
        if not self.bids:
            return None
        return -self.bids[0][0]  # Convert negative back to positive

    def get_best_ask(self) -> Optional[float]:
        """Get lowest ask price"""
        if not self.asks:
            return None
        return self.asks[0][0]

    def get_order_book_depth(self, levels: int = 5) -> dict:
        """Get top N levels of order book"""
        # Group orders by price
        bid_levels = {}
        for _, _, order in self.bids[:levels*2]:  # Get more than needed for aggregation
            if order.order_type == OrderType.BID:
                if order.price not in bid_levels:
                    bid_levels[order.price] = 0
                bid_levels[order.price] += order.quantity

        ask_levels = {}
        for _, _, order in self.asks[:levels*2]:
            if order.order_type == OrderType.ASK:
                if order.price not in ask_levels:
                    ask_levels[order.price] = 0
                ask_levels[order.price] += order.quantity

        # Sort and limit
        sorted_bids = sorted(bid_levels.items(), reverse=True)[:levels]
        sorted_asks = sorted(ask_levels.items())[:levels]

        return {
            'bids': sorted_bids,
            'asks': sorted_asks
        }

    def cancel_order(self, order_id: int) -> bool:
        """Cancel an order by ID"""
        if order_id not in self.order_map:
            return False

        order, order_type = self.order_map[order_id]

        if order_type == OrderType.BID:
            # Find and remove from bids heap (lazy deletion)
            for i, (neg_price, oid, _) in enumerate(self.bids):
                if oid == order_id:
                    self.bids[i] = self.bids[-1]
                    self.bids.pop()
                    heapq.heapify(self.bids)  # Re-heapify
                    break
        else:
            # Find and remove from asks heap
            for i, (price, oid, _) in enumerate(self.asks):
                if oid == order_id:
                    self.asks[i] = self.asks[-1]
                    self.asks.pop()
                    heapq.heapify(self.asks)
                    break

        del self.order_map[order_id]
        return True

class ListOrderBook:
    """Original implementation using lists with manual sorting"""
    def __init__(self):
        self.bids = []  # List of (price, quantity, order_id)
        self.asks = []
        self.order_map = {}
        self.next_order_id = 1

    def add_order(self, price: float, quantity: int, order_type: OrderType) -> int:
        order_id = self.next_order_id
        self.next_order_id += 1

        if order_type == OrderType.BID:
            self.bids.append((price, quantity, order_id))
            # Sort bids descending by price
            self.bids.sort(key=lambda x: (-x[0], x[2]))
        else:
            self.asks.append((price, quantity, order_id))
            # Sort asks ascending by price
            self.asks.sort(key=lambda x: (x[0], x[2]))

        self.order_map[order_id] = (price, quantity, order_type)
        self._match_orders_list()
        return order_id

    def _match_orders_list(self):
        while self.bids and self.asks:
            best_bid_price, best_bid_qty, bid_id = self.bids[0]
            best_ask_price, best_ask_qty, ask_id = self.asks[0]

            if best_bid_price >= best_ask_price:
                matched_qty = min(best_bid_qty, best_ask_qty)

                # Update quantities
                if best_bid_qty == matched_qty:
                    self.bids.pop(0)
                    del self.order_map[bid_id]
                else:
                    self.bids[0] = (best_bid_price, best_bid_qty - matched_qty, bid_id)

                if best_ask_qty == matched_qty:
                    self.asks.pop(0)
                    del self.order_map[ask_id]
                else:
                    self.asks[0] = (best_ask_price, best_ask_qty - matched_qty, ask_id)
            else:
                break

    def get_best_bid(self) -> Optional[float]:
        return self.bids[0][0] if self.bids else None

    def get_best_ask(self) -> Optional[float]:
        return self.asks[0][0] if self.asks else None

def generate_orders(n: int = 10000, spread: float = 1.0) -> List[Tuple]:
    """Generate random orders around a mid-price"""
    orders = []
    mid_price = 100.0

    for i in range(n):
        is_bid = random.random() > 0.5
        if is_bid:
            # Bid: price slightly below to above mid
            price = mid_price - random.random() * spread + random.random() * 0.5
            order_type = OrderType.BID
        else:
            # Ask: price slightly above mid
            price = mid_price + random.random() * spread + random.random() * 0.5
            order_type = OrderType.ASK

        quantity = random.randint(1, 100)
        orders.append((price, quantity, order_type))

    return orders

def benchmark_performance():
    """Benchmark heap vs list implementations"""
    orders = generate_orders(10000)

    print("=" * 60)
    print("BENCHMARK: 10,000 ORDER INSERTIONS")
    print("=" * 60)

    # Test Heap Implementation
    heap_book = HeapOrderBook()
    heap_times = []

    start_time = time.perf_counter()
    for price, quantity, order_type in orders:
        heap_start = time.perf_counter()
        heap_book.add_order(price, quantity, order_type)
        heap_times.append(time.perf_counter() - heap_start)
    heap_total = time.perf_counter() - start_time

    # Test List Implementation
    list_book = ListOrderBook()
    list_times = []

    start_time = time.perf_counter()
    for price, quantity, order_type in orders:
        list_start = time.perf_counter()
        list_book.add_order(price, quantity, order_type)
        list_times.append(time.perf_counter() - list_start)
    list_total = time.perf_counter() - start_time

    # Statistics
    print(f"\nHEAP IMPLEMENTATION:")
    print(f"  Total time: {heap_total:.4f} seconds")
    print(f"  Average per order: {heap_total/10000*1000:.3f} ms")
    print(f"  Max insertion time: {max(heap_times)*1000:.3f} ms")
    print(f"  Min insertion time: {min(heap_times)*1000:.3f} ms")

    print(f"\nLIST IMPLEMENTATION:")
    print(f"  Total time: {list_total:.4f} seconds")
    print(f"  Average per order: {list_total/10000*1000:.3f} ms")
    print(f"  Max insertion time: {max(list_times)*1000:.3f} ms")
    print(f"  Min insertion time: {min(list_times)*1000:.3f} ms")

    print(f"\nPERFORMANCE COMPARISON:")
    print(f"  Heap is {list_total/heap_total:.2f}x faster")
    print(f"  Time saved: {(list_total - heap_total)*1000:.1f} ms")
    print(f"  Heap efficiency gain: {(1 - heap_total/list_total)*100:.1f}%")

    # Memory usage comparison
    import sys
    heap_memory = sys.getsizeof(heap_book.bids) + sys.getsizeof(heap_book.asks)
    list_memory = sys.getsizeof(list_book.bids) + sys.getsizeof(list_book.asks)

    print(f"\nMEMORY USAGE:")
    print(f"  Heap structure: {heap_memory:,} bytes")
    print(f"  List structure: {list_memory:,} bytes")
    print(f"  Memory ratio: {heap_memory/list_memory:.2f}")

    return heap_total, list_total

def demonstrate_order_book_functionality():
    """Show the order book in action"""
    print("\n" + "=" * 60)
    print("ORDER BOOK FUNCTIONALITY DEMONSTRATION")
    print("=" * 60)

    book = HeapOrderBook()

    # Add some sample orders
    orders = [
        (100.5, 50, OrderType.BID),   # Bid at 100.5
        (101.0, 30, OrderType.BID),   # Bid at 101.0 (best bid)
        (99.5, 40, OrderType.ASK),    # Ask at 99.5 (best ask)
        (100.0, 20, OrderType.ASK),   # Ask at 100.0
        (101.5, 60, OrderType.BID),   # Bid at 101.5 (new best bid)
    ]

    print("\nAdding orders:")
    for price, qty, order_type in orders:
        order_id = book.add_order(price, qty, order_type)
        print(f"  {order_type.value} @ ${price:.2f} x {qty} (ID: {order_id})")
        print(f"    Best Bid: {book.get_best_bid()}, Best Ask: {book.get_best_ask()}")

    print("\nOrder Book Depth (Top 3):")
    depth = book.get_order_book_depth(levels=3)
    print("  BIDS:")
    for price, qty in depth['bids']:
        print(f"    ${price:.2f}: {qty}")
    print("  ASKS:")
    for price, qty in depth['asks']:
        print(f"    ${price:.2f}: {qty}")

def analyze_time_complexity():
    """Explain the time complexity difference"""
    print("\n" + "=" * 60)
    print("TIME COMPLEXITY ANALYSIS")
    print("=" * 60)

    print("\nLIST IMPLEMENTATION:")
    print("  • Insertion: O(n log n) for sorting after each insert")
    print("  • Matching: O(1) for accessing top orders")
    print("  • Total per order: O(n log n)")
    print("  • For 10,000 orders: ~O(10,000 * log(10,000)) ≈ O(138,000)")

    print("\nHEAP IMPLEMENTATION:")
    print("  • Insertion: O(log n) for heap push")
    print("  • Matching: O(log n) for heap pop if match occurs")
    print("  • Total per order: O(log n)")
    print("  • For 10,000 orders: ~O(10,000 * log(10,000)) ≈ O(138,000)")
    print("     But log(n) operations are MUCH faster than n log(n) sorting")

    print("\nWHY HEAP IS FASTER:")
    print("  1. Heap maintains partial order, not full sort")
    print("  2. Heap operations use binary tree structure")
    print("  3. Python's heapq is implemented in C (optimized)")
    print("  4. No need to rearrange entire list on each insert")

if __name__ == "__main__":
    # Run demonstration
    demonstrate_order_book_functionality()

    # Run benchmark
    heap_time, list_time = benchmark_performance()

    # Show complexity analysis
    analyze_time_complexity()

    print("\n" + "=" * 60)
    print("KEY INSIGHTS")
    print("=" * 60)
    print("""
    1. Heap-based order book provides O(log n) insertions vs O(n log n) for lists
    2. The performance difference becomes dramatic with large order volumes
    3. Heap is optimal for order books because we only need the 'best' orders
    4. Real exchanges use similar heap-based structures for matching engines
    5. The heap implementation scales better for high-frequency trading
    """)


ORDER BOOK FUNCTIONALITY DEMONSTRATION

Adding orders:
  BID @ $100.50 x 50 (ID: 1)
    Best Bid: 100.5, Best Ask: None
  BID @ $101.00 x 30 (ID: 2)
    Best Bid: 101.0, Best Ask: None
  ASK @ $99.50 x 40 (ID: 3)
    Best Bid: 100.5, Best Ask: None
  ASK @ $100.00 x 20 (ID: 4)
    Best Bid: 100.5, Best Ask: None
  BID @ $101.50 x 60 (ID: 5)
    Best Bid: 101.5, Best Ask: None

Order Book Depth (Top 3):
  BIDS:
    $101.50: 60
    $100.50: 20
  ASKS:
BENCHMARK: 10,000 ORDER INSERTIONS

HEAP IMPLEMENTATION:
  Total time: 0.0178 seconds
  Average per order: 0.002 ms
  Max insertion time: 0.413 ms
  Min insertion time: 0.001 ms

LIST IMPLEMENTATION:
  Total time: 3.7847 seconds
  Average per order: 0.378 ms
  Max insertion time: 40.397 ms
  Min insertion time: 0.001 ms

PERFORMANCE COMPARISON:
  Heap is 212.19x faster
  Time saved: 3766.9 ms
  Heap efficiency gain: 99.5%

MEMORY USAGE:
  Heap structure: 74,416 bytes
  List structure: 74,416 bytes
  Memory ratio: 1.00

TIME COMPLEXITY ANAL